# 📘 S9_P3 — PyTorch : les fondamentaux, brique par brique

> **Comment lire ce notebook.** Le code est **exactement celui que tu as écrit** — je n'ai
> touché à aucune ligne. J'ai seulement intercalé des explications entre tes cellules.
> Ton notebook d'origine est dans `01_Exercices_originaux/`.

## 🎯 Où tu en es

Depuis S1, tu faisais du **scikit-learn** : tu choisissais un modèle tout fait, tu appelais
`.fit()`, c'était réglé. PyTorch renverse la table : tu **construis** le modèle et tu
**écris** l'entraînement. Plus long, mais c'est le seul moyen de dépasser les modèles
standards.

Ce notebook monte l'escalier marche par marche. Chaque étage sert au suivant :

| Étage | Ce que tu apprends | Cellules |
|---|---|---|
| 1️⃣ **Le tenseur** | la donnée de base, `.shape` | 2 → 11 |
| 2️⃣ **Le device** | envoyer un calcul sur le GPU | 12 → 15 |
| 3️⃣ **L'autograd** | la dérivée automatique — *le cœur du deep learning* | 16 → 21 |
| 4️⃣ **`nn.Module`** | assembler des couches en réseau | 23 → 25 |
| 5️⃣ **`Dataset` / `DataLoader`** | servir les données par lots | 26 → 33 |
| 6️⃣ **La boucle d'entraînement** | à la main, pour comprendre | 30, 34 |
| 7️⃣ **Lightning** | la même boucle, écrite pour toi | 35 → 38 |

Les étages 3 et 6 sont ceux qui comptent. Le reste est de la plomberie.

In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from torchmetrics import MeanSquaredError
import matplotlib.pyplot as plt

In [2]:
pl.seed_everything(32)

Seed set to 32


32

## 🧠 THÉORIE — Le tenseur

Un **tenseur** est un tableau de nombres à N dimensions. C'est *la seule* structure de
données que PyTorch manipule. Le vocabulaire change avec le nombre de dimensions :

| Nom | Dimensions | `.shape` | Exemple concret |
|---|---|---|---|
| **scalaire** | 0 | `torch.Size([])` | une perte : `3.0` |
| **vecteur** | 1 | `torch.Size([2])` | les features d'un échantillon |
| **matrice** | 2 | `torch.Size([2, 2])` | un batch : `(lignes, colonnes)` |
| **tenseur 3D** | 3 | `torch.Size([8, 28, 28])` | 8 images de 28×28 |
| **tenseur 4D** | 4 | `torch.Size([8, 3, 28, 28])` | 8 images **couleur** |

### ⚠️ `.shape` est ton outil de débogage n°1

Retiens ça maintenant, ça t'évitera des heures : **la grande majorité des erreurs PyTorch
sont des dimensions qui ne collent pas**. Un message comme
`mat1 and mat2 shapes cannot be multiplied (32x2 and 8x1)` veut juste dire « tu as branché
un tuyau de 2 sur une entrée de 8 ».

Le réflexe : dès que ça coince, `print(x.shape)` avant et après chaque étape.

In [3]:
scalar = torch.tensor(3.0)
vector = torch.tensor([3.0, 1.0])
matrix = torch.tensor([[3.0, 1.0], [3.0, 1.0]])
print(f"scaler {scalar} shape {scalar.shape}")
print(f"vector {vector} shape {vector.shape}")
print(f"matrix {matrix} shape {matrix.shape}")

scaler 3.0 shape torch.Size([])
vector tensor([3., 1.]) shape torch.Size([2])
matrix tensor([[3., 1.],
        [3., 1.]]) shape torch.Size([2, 2])


In [4]:
# Create and visualize the values and size of a matrix 3 rows and 2 columns

## 🧠 THÉORIE — NumPy vs torch : pourquoi une deuxième bibliothèque ?

Les cellules qui suivent font **exactement le même calcul** avec NumPy et avec torch, et
affichent le même résultat. Alors pourquoi torch existe-t-il ?

Trois raisons, et **seule la troisième compte vraiment** :

| | NumPy | torch |
|---|---|---|
| Calcul vectorisé | ✅ | ✅ |
| Tourne sur **GPU** | ❌ CPU uniquement | ✅ `.to("cuda")` |
| **Calcule les dérivées tout seul** | ❌ | ✅ **`autograd`** |

L'autograd (étage 3, dans quelques cellules) est **la** raison d'être de PyTorch. Sans lui,
entraîner un réseau voudrait dire dériver à la main des milliers de paramètres. Personne ne
fait ça.

L'API est volontairement calquée sur NumPy pour que tu ne réapprennes rien : `*`, `+`,
`.mean()` se comportent pareil, **broadcasting compris**.

In [5]:
np_array = np.array([1.0, 2.0, 3.0])
torch_tensor = torch.tensor([1.0, 2.0, 3.0])

In [6]:
print(f" Numpy result {np_array * 2}")
print(f" Tensor result {torch_tensor * 2}")

 Numpy result [2. 4. 6.]
 Tensor result tensor([2., 4., 6.])


## Multiply a and b element by element

In [7]:
print(f" Numpy result {np_array * np_array}")
print(f" Tensor result {torch_tensor * torch_tensor}")

 Numpy result [1. 4. 9.]
 Tensor result tensor([1., 4., 9.])


## Add a and b

In [8]:
print(f" Numpy result {np_array + np_array}")
print(f" Tensor result {torch_tensor + torch_tensor}")

 Numpy result [2. 4. 6.]
 Tensor result tensor([2., 4., 6.])


## Mean of a

In [9]:
print(f" Numpy result {np_array.mean()}")
print(f" Tensor result {torch_tensor.mean()}")

 Numpy result 2.0
 Tensor result 2.0


## 🧠 THÉORIE — Le device : où le calcul se passe

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

Cette ligne est **le patron standard** de tout code PyTorch. Elle dit : « prends le GPU s'il
y en a un, sinon débrouille-toi avec le CPU ». Écrit comme ça, ton notebook tourne partout —
ton PC, celui d'un pote sans carte, un T4 chez Google — **sans changer une ligne**.

### 🔴 Le piège à connaître : `cuda` ne veut pas dire NVIDIA

Sur ton Radeon 880M, `torch.cuda.is_available()` renvoie **`True`**. Surprenant ? CUDA est
la techno de NVIDIA, et tu es en AMD. C'est parce que **ROCm** (la couche AMD) réutilise
volontairement les *noms* de l'API CUDA : ton code écrit pour NVIDIA tourne tel quel chez
AMD. Le nom `cuda` désigne ici « l'accélérateur », pas la marque.

### ⚠️ Rien ne part sur le GPU tout seul

Un tenseur créé normalement vit en **RAM** (côté CPU). C'est **`.to(device)` et lui seul**
qui le copie sur la carte. Si tu oublies, tu récoltes :

```
RuntimeError: Expected all tensors to be on the same device
```

La règle : **le modèle et les données doivent être sur le même device.** Les cellules
suivantes te font la démonstration — `x_cpu`, puis `x_device`, puis `.device` pour vérifier
où il a atterri.

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device {device}")

Selected device cuda


In [11]:
x_cpu = torch.tensor([1.0, 2.0, 3.0])
device

device(type='cuda')

In [12]:
x_device = x_cpu.to(device)
x_device

tensor([1., 2., 3.], device='cuda:0')

In [13]:
x_device.device

device(type='cuda', index=0)

## 🧠 THÉORIE — L'autograd : le cœur du deep learning

**Si tu ne dois retenir qu'une chose de ce notebook, c'est cette section.**

### Le problème

Entraîner un réseau, c'est chercher les paramètres qui minimisent l'erreur. Pour savoir dans
quel sens pousser chaque paramètre, il faut la **dérivée** de l'erreur par rapport à ce
paramètre. Un vrai réseau en a des millions. Les dériver à la main est impossible.

### La solution : PyTorch les calcule pour toi

```python
x = torch.tensor(2.0, requires_grad=True)
```

`requires_grad=True` dit à PyTorch : « **surveille ce tenseur** ». À partir de là, chaque
opération qu'il subit est enregistrée dans un **graphe de calcul**.

Quand tu écris `y = x**2 + 3*x + 1`, PyTorch ne calcule pas seulement `y = 11`. Il retient
aussi *comment* il y est arrivé : un carré, une multiplication, deux additions.

### `.backward()` : remonter le graphe

`y.backward()` parcourt ce graphe **à l'envers** en appliquant la règle de dérivation en
chaîne, et dépose le résultat dans `x.grad`.

### 📐 Vérifions à la main

$$y = x^2 + 3x + 1 \qquad \Rightarrow \qquad \frac{dy}{dx} = 2x + 3$$

En $x = 2$ : $\frac{dy}{dx} = 2(2) + 3 = \mathbf{7}$

C'est **exactement** ce que fait ta cellule suivante : tu calcules `dy = 2*x + 3` toi-même,
puis tu lances `y.backward()` et tu compares avec `x.grad.item()`. Les deux donnent `7.0`.

> **C'est toute la magie.** Ton `dy = 2*x + 3`, tu as dû le dériver dans ta tête. PyTorch,
> lui, n'a **jamais vu** la formule de la dérivée — il l'a reconstruite depuis le graphe.
> Change `y` en `sin(x) * exp(x)`, il suivra sans broncher. C'est ce qui rend possible
> l'entraînement de réseaux à des millions de paramètres.

In [14]:
x = torch.tensor(2.0, requires_grad=True)

In [15]:
y = x**2 + 3*x+1
y

tensor(11., grad_fn=<AddBackward0>)

In [16]:
dy = 2*x + 3
dy

tensor(7., grad_fn=<AddBackward0>)

In [17]:
y.backward()

In [18]:
print(f"x: {x}, y: {y}")
print(f"dy/dx at x=2 {x.grad.item()}")

x: 2.0, y: 11.0
dy/dx at x=2 7.0


### ✍️ À toi — la dérivée de $x^4 + 4$

Même exercice, fonction différente :

$$y = x^4 + 4 \qquad \Rightarrow \qquad \frac{dy}{dx} = 4x^3$$

En $x = 4$ : $4 \times 4^3 = 4 \times 64 = \mathbf{256}$

Note que le `+ 4` disparaît : la dérivée d'une constante est nulle. Vérifie que `x.grad`
affiche bien `256.0`.

> ⚠️ **Le piège des gradients qui s'accumulent.** Ici tu redéfinis `x` avec un nouveau
> `torch.tensor(...)`, donc son `.grad` repart de zéro. Mais si tu appelais `.backward()`
> **deux fois sur le même `x`**, PyTorch **additionnerait** les gradients au lieu de les
> remplacer. C'est voulu (utile pour certaines techniques), et c'est exactement pour ça que
> toute boucle d'entraînement contient un `optimizer.zero_grad()` — tu le verras plus bas.

In [19]:
x = torch.tensor(4.0, requires_grad=True)
y = x ** 4 +4
y.backward()
print(f"dy/dx at x = 4.0 {x.grad.item()}")

dy/dx at x = 4.0 256.0


## Neural Networks

## 🧠 THÉORIE — `nn.Module` : assembler un réseau

Tout réseau PyTorch est une classe qui **hérite de `nn.Module`** et remplit deux méthodes :

| Méthode | Rôle | Quand elle tourne |
|---|---|---|
| `__init__` | **déclare** les couches | une fois, à la création |
| `forward` | **enchaîne** les couches | à chaque prédiction |

### `nn.Linear(in_features=1, out_features=4)`

C'est une couche **dense** : elle applique $y = Wx + b$. Ici elle prend 1 nombre et en
ressort 4. Ses paramètres $W$ (4×1) et $b$ (4) sont créés **automatiquement**, initialisés
au hasard, et déclarés `requires_grad=True` — donc l'autograd les suit déjà.

### `nn.ReLU()` — et pourquoi elle est indispensable

$$\text{ReLU}(x) = \max(0, x)$$

Elle remplace les négatifs par zéro. Ça a l'air trop bête pour servir à quelque chose. C'est
pourtant **ce qui rend le réseau capable d'apprendre autre chose qu'une droite**.

> **Le raisonnement, en une ligne :** deux couches linéaires enchaînées sans rien entre
> elles, ça fait $W_2(W_1x + b_1) + b_2$ — développe, tu retombes sur $W'x + b'$, **une
> seule couche linéaire**. Tes 12 paramètres ne valent pas mieux qu'une régression linéaire
> de S1. La ReLU casse cette linéarité, et c'est **elle** qui donne au réseau sa puissance.

### ⚠️ On appelle `model(x)`, jamais `model.forward(x)`

Ta cellule suivante écrit `model(example_input)` — c'est la bonne façon. `nn.Module` définit
`__call__`, qui appelle `forward` **plus** toute la mécanique interne (hooks, mode train/eval).
Court-circuiter avec `.forward()` marche en apparence et casse des choses en silence.

In [20]:
class TinyNetworks(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=1, out_features=4)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(in_features=4, out_features=1)
    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

In [21]:
model = TinyNetworks()
example_input = torch.tensor([2.0])
example_output = model(example_input)
print(example_output)

tensor([0.4460], grad_fn=<ViewBackward0>)


### 📝 `StudentNetworks` — l'exercice

Même structure que `TinyNetworks`, mais dimensionnée pour **2 features en entrée** et une
couche cachée de **8 neurones**. Compte ses paramètres :

| Couche | Calcul | Paramètres |
|---|---|---|
| `Linear(2, 8)` | 2×8 poids + 8 biais | 24 |
| `Linear(8, 1)` | 8×1 poids + 1 biais | 9 |
| | | **33 au total** |

> Note au passage : tu la définis mais tu ne t'en sers jamais dans la suite du notebook.
> Ce n'est pas une erreur — c'était l'exercice. C'est `TinyNetworks` qui travaille ensuite.

In [22]:
class StudentNetworks(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=2, out_features=8)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(in_features=8, out_features=1)
    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

## 🧠 THÉORIE — Des données jouet, choisies exprès

```python
x = [[0.], [1.], [2.], [3.]]
y = [[1.], [2.], [4.], [6.]]
```

Quatre points, une relation quasi affine ($y \approx 2x$, à peu près). C'est **volontairement
minuscule** : tu dois pouvoir vérifier chaque étape mentalement. Si ton réseau n'arrive pas à
apprendre *ça*, inutile de le lancer sur des vraies données.

Remarque les **doubles crochets** : la forme est `(4, 1)` et non `(4,)`. PyTorch veut
toujours `(nombre d'échantillons, nombre de features)`, même quand il n'y a qu'une feature.
C'est la source d'erreur de forme n°1 chez les débutants.

In [23]:
x = torch.tensor([[0.0], [1.0], [2.0], [3.0]])
y = torch.tensor([[1.0], [2.0], [4.0], [6.0]])

## 🧠 THÉORIE — `Dataset` : le contrat

Un `Dataset` PyTorch est une classe qui promet **trois** méthodes. C'est un contrat :

| Méthode | Ce qu'elle doit répondre |
|---|---|
| `__init__` | charge ou fabrique les données |
| `__len__` | « combien d'échantillons ? » |
| `__getitem__(i)` | « donne-moi **l'échantillon i** » |

Respecte le contrat, et tout l'écosystème PyTorch sait travailler avec ta classe — le
`DataLoader`, Lightning, tout.

### ⚠️ `__getitem__` doit **retourner** quelque chose

C'est **l'erreur classique**, et elle est vicieuse : si tu oublies le `return`, Python
renvoie `None` sans se plaindre, et tu récoltes un message incompréhensible bien plus loin.
Ici ton `return self.x[index], self.y[index]` est correct.

> Le mot **index** est important : `__getitem__` sert **un** échantillon, pas un lot. C'est
> le `DataLoader` qui regroupera. Chacun son travail.

In [24]:
class TinyDataset(Dataset):
    def __init__(self):
        self.x = torch.tensor([[0.0], [1.0], [2.0], [3.0]])
        self.y = torch.tensor([[1.0], [2.0], [4.0], [6.0]])
    def __len__(self):
        return len(self.x)
    def __getitem__(self, index):
        return self.x[index], self.y[index]

dataset = TinyDataset()
len(dataset)

# for batch in dataset:
#     x, y = batch
#     print(f"x: {x} - y: {y}")

4

## 🧠 THÉORIE — `DataLoader` : l'emballeur

Le `Dataset` sait servir **un** échantillon. Le `DataLoader` l'enveloppe et gère tout le
reste :

| Argument | Rôle | Pourquoi |
|---|---|---|
| `batch_size=2` | regroupe par lots de 2 | on met à jour les poids par lot, pas par point |
| `shuffle=True` | mélange à chaque epoch | sinon le réseau apprend **l'ordre** des données |
| `num_workers` | charge en parallèle | utile sur gros datasets |

### Pourquoi des batches ?

Trois raisons : ça tient en mémoire, ça vectorise (donc c'est rapide), et le bruit d'un
petit lot **aide** à sortir des minima locaux. Le batch est un curseur entre stabilité et
vitesse.

### ⚠️ `shuffle=True` sur l'entraînement, `False` sur la validation

Toujours. Mélanger la validation ne sert à rien et rend tes résultats non reproductibles.
Lightning te le rappellera avec un warning si tu l'oublies.

In [25]:
loader = DataLoader(dataset, batch_size=2, shuffle=True)

### 🔴 ATTENTION — cette cellule contient une erreur (instructive)

```python
for batch_x, batch_y in enumerate(loader):
```

`enumerate` produit des paires **`(indice, élément)`**. Donc ici :

- `batch_x` reçoit **l'indice du lot** — un entier : `0`, `1`…
- `batch_y` reçoit **le lot entier** — la liste `[x, y]`

Tes noms de variables mentent : `batch_x` n'est pas un x, c'est un compteur. Regarde ta
sortie, tu verras `batch_x = 0` et non un tenseur.

**Les deux écritures correctes**, au choix :

```python
for batch_x, batch_y in loader:              # sans enumerate : le loader donne deja (x, y)
    ...

for i, (batch_x, batch_y) in enumerate(loader):   # avec enumerate : il faut les parentheses
    ...
```

> 👉 **Tu l'as corrigé toi-même plus loin** : va voir ta cellule `for i, (batch_x, batch_y)
> in enumerate(loader)` — parenthèses comprises. C'est la bonne version. Je laisse celle-ci
> telle quelle, c'est ton notebook original et l'erreur est pédagogique.

In [26]:
for batch_x, batch_y in enumerate(loader):
    print(f"batch_x = {batch_x} - batch_y {batch_y}")

batch_x = 0 - batch_y [tensor([[0.],
        [1.]]), tensor([[1.],
        [2.]])]
batch_x = 1 - batch_y [tensor([[2.],
        [3.]]), tensor([[4.],
        [6.]])]


## 🧠 THÉORIE — La boucle d'entraînement, à la main

**La section la plus importante du notebook.** Ces cinq lignes sont le moteur de *tout* le
deep learning. Lightning va bientôt les écrire à ta place — mais tu dois savoir ce qu'il
cache.

```python
prediction = pure_model(batch_x)      # 1. FORWARD  : predire
loss = loss_fn(prediction, batch_y)   # 2. LOSS     : mesurer l'erreur
optimzer.zero_grad()                  # 3. RAZ      : effacer les gradients precedents
loss.backward()                       # 4. BACKWARD : calculer les derivees
optimzer.step()                       # 5. STEP     : corriger les poids
```

| Étape | Ce qui se passe | Si tu l'oublies |
|---|---|---|
| **forward** | les données traversent le réseau | — |
| **loss** | `MSELoss` = moyenne des $(\hat{y} - y)^2$ | — |
| **zero_grad** | remet `.grad` à zéro | 🔴 les gradients **s'additionnent** → n'importe quoi |
| **backward** | l'autograd remplit chaque `.grad` | 🔴 rien à corriger |
| **step** | $w \leftarrow w - \text{lr} \times \text{grad}$ | 🔴 les poids ne bougent **jamais** |

### 📐 La descente de gradient, en une formule

$$w_{\text{nouveau}} = w_{\text{ancien}} - \eta \cdot \frac{\partial L}{\partial w}$$

Le gradient pointe vers la **montée** de l'erreur. On part donc dans le sens **opposé** (le
signe `−`), d'un pas proportionnel au **learning rate** $\eta$ (ton `lr=0.01`).

Trop grand, tu sautes par-dessus le minimum et la perte explose. Trop petit, tu n'arrives
jamais. C'est l'hyperparamètre n°1 à régler.

### ⚠️ Trois détails dans **ta** cellule

**1. `model.train()` porte sur le mauvais modèle.** Tu entraînes `pure_model`, mais tu
appelles `model.train()` — `model`, c'est le `TinyNetworks()` que tu as créé plus haut pour
tester une prédiction. Ça ne casse rien ici (`.train()` ne fait que basculer un drapeau, et
il n'y a ni Dropout ni BatchNorm dans ce réseau), mais la ligne ne sert à rien. Ce serait
`pure_model.train()`.

**2. Le `print` dans la boucle interne** affiche chaque prédiction de chaque batch × 60
epochs. C'est parfait pour *voir* ce qui se passe une fois, et à supprimer dès que ça marche.

**3. Tu affiches `loss.item()`, pas la moyenne.** Tu calcules pourtant `average_total_error`
juste au-dessus… et tu ne l'affiches pas. Du coup tu lis l'erreur du **dernier batch**, qui
saute d'un epoch à l'autre. 👉 Tu as corrigé ça dans ta deuxième boucle : elle affiche bien
`average_total_error`.

In [27]:
pure_model = TinyNetworks()
loss_fn = nn.MSELoss()
optimzer = torch.optim.SGD(pure_model.parameters(), lr=0.01)

for epoch in range(60):
    model.train()
    total_train_loss = 0
    for i, (batch_x, batch_y) in enumerate(loader):
        prediction = pure_model(batch_x)
        print(f"inputs: {batch_x}\t predictions : {prediction}")
        loss = loss_fn(prediction, batch_y)
        optimzer.zero_grad()
        loss.backward()
        optimzer.step()
        total_train_loss += loss.item() * batch_x.size(0)
    average_total_error = total_train_loss / len(loader.dataset)
    print(f"epoch={epoch}, loss={loss.item():.4f}")

inputs: tensor([[0.],
        [2.]])	 predictions : tensor([[-0.0571],
        [-0.3254]], grad_fn=<AddmmBackward0>)
inputs: tensor([[3.],
        [1.]])	 predictions : tensor([[-0.4538],
        [ 0.0636]], grad_fn=<AddmmBackward0>)
epoch=0, loss=22.7004
inputs: tensor([[2.],
        [0.]])	 predictions : tensor([[0.1255],
        [0.1135]], grad_fn=<AddmmBackward0>)
inputs: tensor([[1.],
        [3.]])	 predictions : tensor([[0.2834],
        [0.0490]], grad_fn=<AddmmBackward0>)
epoch=1, loss=19.1809
inputs: tensor([[0.],
        [1.]])	 predictions : tensor([[0.2526],
        [0.3634]], grad_fn=<AddmmBackward0>)
inputs: tensor([[3.],
        [2.]])	 predictions : tensor([[0.2781],
        [0.3958]], grad_fn=<AddmmBackward0>)
epoch=2, loss=22.8651
inputs: tensor([[3.],
        [1.]])	 predictions : tensor([[0.4864],
        [0.4925]], grad_fn=<AddmmBackward0>)
inputs: tensor([[0.],
        [2.]])	 predictions : tensor([[0.4536],
        [0.5933]], grad_fn=<AddmmBackward0>)
epoch=3, l

## 📈 Deuxième manche — un vrai dataset

`TinyDataset2` change d'échelle : **2000 points** au lieu de 4, et une relation **exactement**
linéaire ($y = 2x$), sans bruit.

```python
self.x = torch.linspace(start=0, end=100, steps=n_samples).reshape(-1, 1)
self.y = self.x * 2
```

`linspace(0, 100, 2000)` étale 2000 valeurs régulières entre 0 et 100. Le `.reshape(-1, 1)`
transforme `(2000,)` en `(2000, 1)` — encore la forme `(échantillons, features)`. Le `-1`
signifie « calcule cette dimension toute seule ».

### ⚠️ Des données non normalisées, exprès

Tes `x` vont **jusqu'à 100**. Souviens-toi de S1 : on normalise toujours. Ici, non — et tu
vas le sentir. Avec des entrées à 100 et `lr=0.01`, les gradients sont énormes et
l'entraînement part en vrille (perte à `NaN` ou qui explose).

C'est **la leçon de la cellule** : le deep learning ne dispense **pas** de préparer les
données. Le S9_P4 mettra un `StandardScaler` avant le réseau, et tout ira bien.

In [28]:
class TinyDataset2(Dataset):
    def __init__(self, n_samples=1000):
        self.x = torch.linspace(start=0, end=100, steps=n_samples).reshape(-1, 1)
        self.y = self.x * 2
    def __len__(self):
        return len(self.x)
    def __getitem__(self, index):
        return self.x[index], self.y[index]

dataset = TinyDataset2(n_samples=2000)
len(dataset)

2000

In [29]:
loader = DataLoader(dataset, batch_size=200, shuffle=True)

In [30]:
for i, (batch_x, batch_y) in enumerate(loader):
    print(f"iteration = {i}  batch_x {batch_x[0]} batch len = {len(batch_x)}")

iteration = 0  batch_x tensor([10.3552]) batch len = 200
iteration = 1  batch_x tensor([45.7729]) batch len = 200
iteration = 2  batch_x tensor([55.5278]) batch len = 200
iteration = 3  batch_x tensor([23.6618]) batch len = 200
iteration = 4  batch_x tensor([37.4687]) batch len = 200
iteration = 5  batch_x tensor([8.4042]) batch len = 200
iteration = 6  batch_x tensor([40.7204]) batch len = 200
iteration = 7  batch_x tensor([84.4922]) batch len = 200
iteration = 8  batch_x tensor([7.4537]) batch len = 200
iteration = 9  batch_x tensor([28.7644]) batch len = 200


### La même boucle, en mieux

Deux différences avec la première :

- ✅ `for i, (batch_x, batch_y) in enumerate(loader)` — la bonne syntaxe, parenthèses comprises
- ✅ `print(f"... loss={average_total_error:.4f}")` — la **moyenne** de l'epoch, pas le dernier batch

C'est la version à retenir. (Le `model.train()` au lieu de `pure_model.train()` est toujours là.)

In [31]:
pure_model = TinyNetworks()
loss_fn = nn.MSELoss()
optimzer = torch.optim.SGD(pure_model.parameters(), lr=0.01)

for epoch in range(200):
    model.train()
    total_train_loss = 0
    for i, (batch_x, batch_y) in enumerate(loader):
        prediction = pure_model(batch_x)
        print(f"inputs: {batch_x}\tpredictions : {prediction}")
        loss = loss_fn(prediction, batch_y)
        optimzer.zero_grad()
        loss.backward()
        optimzer.step()
        total_train_loss += loss.item() * batch_x.size(0)
    average_total_error = total_train_loss / len(loader.dataset)
    print(f"epoch={epoch}, loss={average_total_error:.4f}")

[sortie volumineuse retiree : cette boucle imprime chaque tenseur de chaque batch
 sur 200 epochs (~19 Mo). Relance la cellule pour la reproduire.]


## 🧠 THÉORIE — PyTorch Lightning : la même chose, sans la plomberie

Tu viens d'écrire la boucle deux fois. Tu as vu ce qu'elle contient de répétitif : les
`zero_grad`, les `.item()`, les moyennes, et bientôt `.to(device)`, la validation, les
checkpoints, les logs…

**Lightning écrit tout ça pour toi.** Tu ne fournis que ce qui est *propre à ton problème* :

| Méthode | Ce que tu écris | Ce que Lightning fait |
|---|---|---|
| `__init__` | tes couches, ta loss | — |
| `forward` | ta prédiction | — |
| `training_step` | forward + loss, **et tu retournes la loss** | `zero_grad`, `backward`, `step` |
| `validation_step` | pareil, sans gradient | `torch.no_grad()`, mode eval |
| `configure_optimizers` | quel optimiseur | l'appelle au bon moment |

`self.save_hyperparameters()` enregistre les arguments d'`__init__` dans `self.hparams` —
pratique pour retrouver le `learning_rate` d'un run six mois plus tard.

### 🔴 Deux vraies erreurs dans **ta** cellule

**1. `training_step` ne retourne rien.**

```python
def training_step(self, batch, batch_idx):
    ...
    self.log("train_loss", loss, prog_bar=True)
    # ← il manque : return loss
```

Sans ce `return`, **Lightning n'a aucune loss à propager** : pas de `backward`, pas de
`step`, **les poids ne bougent jamais**. Ton entraînement tournera, affichera une barre de
progression, et n'apprendra strictement rien. Lightning te prévient d'ailleurs :

```
`training_step` returned `None`. If this was on purpose, ignore this warning...
```

**2. `configure_optimizers` optimise le mauvais réseau.**

```python
optimizer = torch.optim.SGD(pure_model.parameters(), lr=self.learning_rate)
#                           ^^^^^^^^^^ devrait etre self.parameters()
```

`pure_model` est le réseau de ta boucle manuelle, **pas** celui de ton `LightningModule`
(qui est `self.network`). Tu corrigerais le premier `return`, l'optimiseur mettrait quand
même à jour un réseau que Lightning n'utilise pas. **Deux bugs qui se cachent l'un l'autre.**

> 👉 Les deux sont corrigés dans ton **S9_P4** : `return loss` présent, et
> `torch.optim.SGD(self.parameters(), ...)`. Compare les deux cellules, c'est la meilleure
> façon de retenir.

In [32]:
class TinyLightningModule(pl.LightningModule):
    def __init__(self, learning_rate=0.01):
        super().__init__()
        self.save_hyperparameters()
        self.network = TinyNetworks()
        self.loss_fn = nn.MSELoss()
        self.val_mse = MeanSquaredError()
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.network(x)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        predictions = self(x)
        loss = self.loss_fn(predictions, y)
        self.log("train_loss", loss, prog_bar=True)

    def validation_step(self, batch, batch_idx):
        x, y = batch
        predictions = self(x)
        loss = self.loss_fn(predictions, y)
        # self.val_mse.update(prediction, y)
        self.log("val_loss", loss, prog_bar=True)
    
    # def on_validation_epoch_end(self):
    #     mse = self.val_mse.compute()
    #     self.log("val_mes", mse, prog_bar=True)
    #     self.val_mse.reset()
    
    def configure_optimizers(self):
        optimizer = torch.optim.SGD(pure_model.parameters(), lr=self.learning_rate)
        return optimizer

In [33]:
lightning_model = TinyLightningModule(learning_rate=0.01)

## 🧠 THÉORIE — Le `Trainer`

```python
trainer = pl.Trainer(max_epochs=30, accelerator="auto", devices="auto",
                     logger=False, enable_checkpointing=False)
```

| Argument | Effet |
|---|---|
| `max_epochs=30` | nombre de passages sur le dataset |
| `accelerator="auto"` | GPU s'il y en a un, CPU sinon — **plus besoin de `.to(device)`** |
| `devices="auto"` | combien d'accélérateurs |
| `logger=False` | pas de fichier de métriques *(le S9_P4 activera `CSVLogger`)* |
| `enable_checkpointing=False` | ne sauvegarde pas les poids |

`accelerator="auto"` est le confort n°1 de Lightning : **il place le modèle et les batches
sur le bon device tout seul**. Tous les `.to(device)` que tu écrivais disparaissent.

### 🔴 Le piège du `Trainer` à usage unique

Un `Trainer` **garde son compteur d'epochs**. Une fois `max_epochs` atteint, rappeler
`.fit()` sur le **même objet** ne fait **rien** : il constate que c'est fini et sort en
`0.0s`, sans lancer un seul batch — pas de barre de progression, juste le résumé du modèle.

Si tu veux réentraîner : **réexécute la cellule qui crée le `Trainer`**, pas seulement le
`.fit()`. (Ou `Restart` + `Run All`.)

In [34]:
trainer = pl.Trainer(max_epochs=30, accelerator="auto", devices="auto", logger=False, enable_checkpointing=False)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


### ▶️ L'entraînement

```python
trainer.fit(lightning_model, train_dataloaders=loader, val_dataloaders=loader)
```

> ⚠️ **`val_dataloaders=loader`, c'est le même loader que l'entraînement.** Tu valides donc
> sur tes données d'entraînement : ta `val_loss` sera la copie de ta `train_loss` et ne
> t'apprendra **rien** sur la généralisation. Sur ce notebook-jouet ça n'a pas d'importance
> (on veut juste voir la mécanique tourner), mais retiens le réflexe : **un jeu de
> validation doit être des données que le modèle n'a jamais vues**. Le S9_P4 fera un vrai
> `train_test_split`.

Vu les deux bugs de la cellule `TinyLightningModule` (pas de `return loss`, mauvais
`.parameters()`), **cet entraînement ne fera rien descendre**. C'est normal, et c'est même
utile : tu vois à quoi ressemble un entraînement qui *a l'air* de marcher mais n'apprend
pas. Une perte parfaitement plate, c'est presque toujours l'un de ces deux bugs.

In [35]:
trainer.fit(lightning_model, train_dataloaders=loader, val_dataloaders=loader)

You are using a CUDA device ('AMD Radeon(TM) 880M Graphics') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | network | TinyNetworks     | 13     | train | 0    
1 | loss_fn | MSELoss          | 0      | train | 0    
2 | val_mse | MeanSquaredError | 0      | train | 0    
-------------------------------------------------------------
13        Trainable params
0         Non-trainable params
13        Total params
0.000     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode
0         Total Flops


c:\Users\kirit\Documents\Projets\pytorch-rocm\venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:485: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
c:\Users\kirit\Documents\Projets\pytorch-rocm\venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.
c:\Users\kirit\Documents\Projets\pytorch-rocm\venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Epoch 0:  50%|█████     | 5/10 [00:00<00:00, 309.21it/s, train_loss=1.95e+4]

c:\Users\kirit\Documents\Projets\pytorch-rocm\venv\Lib\site-packages\pytorch_lightning\loops\optimization\automatic.py:134: `training_step` returned `None`. If this was on purpose, ignore this warning...


Epoch 29: 100%|██████████| 10/10 [00:00<00:00, 149.59it/s, train_loss=2.04e+4, val_loss=1.96e+4]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 10/10 [00:00<00:00, 149.59it/s, train_loss=2.04e+4, val_loss=1.96e+4]


---

# 🎓 Ce qu'il faut retenir du S9_P3

### Les 5 lignes qui font tout le deep learning

```python
prediction = model(batch_x)           # forward
loss = loss_fn(prediction, batch_y)   # erreur
optimizer.zero_grad()                 # RAZ des gradients
loss.backward()                       # autograd : calcule les derivees
optimizer.step()                      # descente de gradient
```

Lightning les écrit pour toi. **Tu dois quand même savoir ce qu'il y a dedans** — le jour où
ça n'apprend pas, c'est là que tu iras chercher.

### La checklist des bugs vus ici

| Symptôme | Cause probable |
|---|---|
| perte **parfaitement plate** | `training_step` sans `return loss` |
| perte plate malgré le `return` | `configure_optimizers` sur les mauvais `.parameters()` |
| `.fit()` en **0.0s**, pas de barre | `Trainer` déjà arrivé à `max_epochs` — recrée-le |
| perte qui **explose** ou `NaN` | données non normalisées (`x` jusqu'à 100) ou `lr` trop grand |
| `batch_x` vaut `0`, `1`, `2`… | `enumerate(loader)` sans les parenthèses |
| `Expected all tensors on same device` | `.to(device)` oublié quelque part |
| dimensions qui ne collent pas | `print(x.shape)` partout, c'est 90 % des erreurs |

### La suite

**S9_P4** reprend tout ça proprement sur un vrai problème de régression : `StandardScaler`,
vrai `train_test_split`, MLP à 2 couches cachées, `CSVLogger` pour tracer les courbes, et
évaluation avec MAE / RMSE / R². C'est le notebook où la mécanique devient un vrai pipeline.